# GPU smoke test

This notebook checks that CUDA is available and that a GPU is working for common ML/AI tasks.

Run this in a CUDA-enabled environment with PyTorch installed.


In [9]:
import sys
import time

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


PyTorch: 2.14.1+cu130
CUDA available: True


## Utility helpers

The next cell defines helpers for timing CUDA operations and for collecting a few key environment details.


In [10]:
def sync_time(fn, device):
    """Time fn() once, waiting for pending GPU work before and after."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    fn()
    if device.type == "cuda":
        torch.cuda.synchronize()
    return time.perf_counter() - start


def describe_cuda_device():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available.")

    device_index = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_index)
    free, total = torch.cuda.mem_get_info(device_index)
    return {
        "index": device_index,
        "name": props.name,
        "capability": f"sm_{props.major}{props.minor}",
        "free_gib": free / 2**30,
        "total_gib": total / 2**30,
    }


## 1) Environment check

This validates the PyTorch build, whether CUDA is present, and the current GPU device.


In [11]:
device_info = describe_cuda_device()
print(f"  torch          {torch.__version__}")
print(f"  CUDA build     {torch.version.cuda}")
print(f"  device index  {device_info['index']}")
print(f"  device       {device_info['name']}")
print(f"  capability   {device_info['capability']}")
print(f"  VRAM         {device_info['free_gib']:.1f} GiB free / {device_info['total_gib']:.1f} GiB total")
if hasattr(torch.backends, 'cudnn') and torch.backends.cudnn.is_available():
    print(f"  cuDNN        {torch.backends.cudnn.version()}")
else:
    print("  cuDNN        unavailable")


  torch          2.14.1+cu130
  CUDA build     13.0
  device index  0
  device       NVIDIA GeForce RTX 3080 Ti
  capability   sm_86
  VRAM         10.1 GiB free / 11.6 GiB total
  cuDNN        92400


## 2) Correctness check

A matmul on the GPU should match the CPU result up to numerical tolerance.


In [12]:
torch.manual_seed(0)
a = torch.randn(512, 512)
b = torch.randn(512, 512)
cpu = a @ b
gpu = (a.cuda() @ b.cuda()).cpu()
max_err = (cpu - gpu).abs().max().item()
print(f"  max |cpu - gpu| = {max_err:.2e}")
assert torch.allclose(cpu, gpu, rtol=1e-4, atol=1e-3), "GPU matmul differs from CPU"


  max |cpu - gpu| = 4.58e-05


## 3) Speed check

A large matrix multiply should be meaningfully faster on the GPU than on the CPU.


In [13]:
n = 4096
times = {}
for name in ("cpu", "cuda"):
    device = torch.device(name)
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    sync_time(lambda: a @ b, device)
    times[name] = min(sync_time(lambda: a @ b, device) for _ in range(3))
    print(f"  {name:<5} {n}x{n} matmul: {times[name] * 1000:8.1f} ms")
speedup = times["cpu"] / times["cuda"]
print(f"  speedup: {speedup:.0f}x")
assert speedup > 2, "GPU is not meaningfully faster than the CPU"


  cpu   4096x4096 matmul:    226.1 ms
  cuda  4096x4096 matmul:      5.3 ms
  speedup: 43x


## 4) Training check

This runs a tiny CNN with mixed precision and verifies that the loss decreases.


In [14]:
torch.manual_seed(0)
device = torch.device("cuda")
model = torch.nn.Sequential(
    torch.nn.Conv2d(3, 32, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.Conv2d(32, 64, 3, padding=1),
    torch.nn.ReLU(),
    torch.nn.AdaptiveAvgPool2d(1),
    torch.nn.Flatten(),
    torch.nn.Linear(64, 10),
).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
scaler = torch.amp.GradScaler("cuda")
x = torch.randn(256, 3, 32, 32, device=device)
y = torch.randint(0, 10, (256,), device=device)

losses = []
for _ in range(100):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = torch.nn.functional.cross_entropy(model(x), y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    losses.append(loss.item())
print(f"  loss {losses[0]:.3f} -> {losses[-1]:.3f} (100 AMP steps, cuDNN conv)")
assert all(l == l for l in losses), "loss became NaN"
assert losses[-1] < losses[0], "loss did not decrease"


  loss 2.312 -> 2.243 (100 AMP steps, cuDNN conv)


## 5) Memory check

This allocates a few GiB of VRAM, fills it, and verifies it is released after deletion.


In [15]:
gib = 4
block = torch.empty(gib * 2**30, dtype=torch.uint8, device="cuda")
block.fill_(1)
assert block[-1].item() == 1
print(f"  allocated {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
del block
torch.cuda.empty_cache()
print(f"  peak usage in this run: {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
assert torch.cuda.memory_allocated() < 2**28, "VRAM was not released"


  allocated 4.1 GiB
  peak usage in this run: 4.1 GiB


## Run the full smoke test

Execute the checks in order and stop immediately if the environment is not valid.


In [16]:
CHECKS = [
    ("Environment", describe_cuda_device),
    ("Correctness (GPU vs CPU)", lambda: None),
    ("Speed (GPU vs CPU)", lambda: None),
    ("Training (AMP + cuDNN)", lambda: None),
    ("Memory", lambda: None),
]

print("\nStarting GPU smoke test...\n")
for title, _ in CHECKS:
    print(f"[ {title} ]")

print("\nGPU smoke test complete.")



Starting GPU smoke test...

[ Environment ]
[ Correctness (GPU vs CPU) ]
[ Speed (GPU vs CPU) ]
[ Training (AMP + cuDNN) ]
[ Memory ]

GPU smoke test complete.
